In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

# always read the latest value from .env
load_dotenv(override=True)

# pool_pre_ping checks the connection is alive before using it
engine = create_engine(os.getenv("DATABASE_URL"), pool_pre_ping=True)

# which project and server? (no password shown)
url = make_url(os.getenv("DATABASE_URL"))
print("User:", url.username)
print("Host:", url.host)

# test + where is the table?
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())
    print(conn.execute(text(
        "SELECT table_schema, table_name FROM information_schema.tables "
        "WHERE table_name = 'air_quality'")).fetchall())

User: postgres.apsdgrcvftokfxkebhin
Host: aws-0-ap-southeast-2.pooler.supabase.com
PostgreSQL 17.11 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit
[]


In [3]:
# load clean data, lowercase column names (simpler in PostgreSQL)
df = pd.read_csv("Downloads/henderson_clean_final_data.csv", parse_dates=["datetime"])
df.columns = [c.lower() for c in df.columns]
print(df.columns.tolist(), len(df))

# upload to a table called air_quality, in chunks so it doesn't time out
df.to_sql("air_quality", engine, if_exists="replace", index=False,
          chunksize=5000, method="multi")

# one row per hour: make datetime the primary key
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE air_quality ADD PRIMARY KEY (datetime)"))

# check
with engine.connect() as conn:
    print(conn.execute(text(
        "SELECT COUNT(*), MIN(datetime), MAX(datetime) FROM air_quality")).fetchone())

['datetime', 'no2', 'pm10', 'rh', 'temp', 'ws'] 150432
(150432, datetime.datetime(2006, 1, 1, 0, 0), datetime.datetime(2023, 2, 28, 23, 0))


In [5]:
# upload predictions table
pred = pd.read_csv("predictions_2019.csv", parse_dates=["datetime"])
pred.to_sql("predictions", engine, if_exists="replace", index=False,
            chunksize=5000, method="multi")

# check
with engine.connect() as conn:
    print(conn.execute(text(
        "SELECT pollutant, model, COUNT(*) FROM predictions GROUP BY pollutant, model")).fetchall())

[('PM10', 'Random Forest', 8671), ('NO2', 'LightGBM', 8461)]


In [7]:
%pip install supabase

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 1.9 MB/s eta 0:00:02
   -------- ------------------------------- 0.8/3.8 MB 1.7 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/3.8 MB 1.6 MB/s eta 0:00:02
   ---------------- ----------------------- 1.6/3.8 MB 1.6 MB/s eta 0:00:02
   --------------------- ------------------ 2.1/3.8 MB 1.7 MB/s eta 0:00:02
   ------------------------ --------------- 2.4/3.8 MB 1.6 MB/s eta 0:00:01
   --------------------------- ------------ 2.6/3.8 MB 1.7 MB/s eta 0:00:01
   -------------------------------- ------- 3.1/3.8 MB 1.6 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 1.6 MB/s eta 0:00:01
   -------------------------------------- - 3.7/3.8 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 1.6 MB/s  0:00:02

   -------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [9]:
from supabase import create_client

# read the API details from .env
load_dotenv(override=True)
sb = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# test 1: read one day of data through the API
res = (sb.table("air_quality")
         .select("*")
         .gte("datetime", "2019-07-01")
         .lt("datetime", "2019-07-02")
         .order("datetime")
         .execute())

api_df = pd.DataFrame(res.data)
print("Rows from API:", len(api_df))
api_df.head()

Rows from API: 24


,datetime,no2,pm10,rh,temp,ws
0,2019-07-01T00:00:00,13.3,16.6,82.9,14,0.8
1,2019-07-01T01:00:00,11.7,12.2,74.8,11,0.4
2,2019-07-01T02:00:00,6.4,11.8,75.8,11,0.6
3,2019-07-01T03:00:00,8.9,13.6,76.3,10,0.6
4,2019-07-01T04:00:00,7.4,1.5,79.2,10,0.5


In [11]:
# test 2: check the API is read-only (this should FAIL)
try:
    sb.table("air_quality").insert({"datetime": "2030-01-01T00:00:00", "pm10": 1}).execute()
    print("WARNING: insert worked - security is not set correctly")
except Exception as e:
    print("Insert blocked (good):", str(e)[:120])

Insert blocked (good): {'message': 'new row violates row-level security policy for table "air_quality"', 'code': '42501', 'hint': None, 'detail
